# 02 - Exploratory Data Analysis
Visualize distributions, test all 10 hypotheses, and identify churn patterns.

## Hypotheses

| # | Hypothesis | Test | Key Column(s) |
|---|-----------|------|---------------|
| H1 | Higher VAN customers are less likely to churn | Mann-Whitney U | VAN |
| H2 | More machines/contracts → lower churn | Mann-Whitney U | Machines, Contracts |
| H3 | Certain case titles predict churn more strongly | Chi-Square | churn_reason_group |
| H4 | Risk type is a strong predictor of churn | Chi-Square | Risk |
| H5 | Larger companies have lower churn rates | Chi-Square + Trend | CompanySize |
| H6 | Higher-tier customers churn less | Chi-Square + Trend | Customer Tier |
| H7 | Proactive case origins lead to lower churn | Chi-Square | Case Origin |
| H8 | Overdue services/repair cases → higher churn | Mann-Whitney U | OverdueServices, Repairs |
| H9 | Full pulls are more associated with churn | Chi-Square | Pull Type |
| H10 | Higher BoB revenue customers churn less | Mann-Whitney U | bob_total_revenue |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
%matplotlib inline

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

from config import PROCESSED_DATA_DIR, NUMERIC_FEATURES
from src.util import load_dataframe

# collect hypothesis results
hypothesis_results = []

In [ ]:
# helper: Mann-Whitney U test for numeric feature vs churn
def test_numeric(df, col, h_id, title):
    churned = df[df['is_churned'] == 1][col].dropna()
    saved = df[df['is_churned'] == 0][col].dropna()
    stat, pval = stats.mannwhitneyu(churned, saved, alternative='two-sided')
    sig = pval < 0.05
    direction = 'Higher in churned' if churned.mean() > saved.mean() else 'Lower in churned'
    print(f'\n{h_id}: {title}')
    print(f'  Churned mean: {churned.mean():.2f} (n={len(churned):,})')
    print(f'  Saved mean:   {saved.mean():.2f} (n={len(saved):,})')
    print(f'  p-value: {pval:.2e} -> {"SIGNIFICANT" if sig else "NOT SIGNIFICANT"}')
    print(f'  Direction: {direction}')
    hypothesis_results.append({'ID': h_id, 'Hypothesis': title, 'Test': 'Mann-Whitney U',
                               'p-value': f'{pval:.2e}', 'Significant': 'Yes' if sig else 'No',
                               'Verdict': 'SUPPORTED' if sig else 'NOT SUPPORTED'})
    return sig

# helper: Chi-Square test for categorical feature vs churn
def test_categorical(df, col, h_id, title):
    ct = pd.crosstab(df[col].fillna('Unknown'), df['is_churned'])
    chi2, pval, dof, expected = stats.chi2_contingency(ct)
    sig = pval < 0.05
    churn_by_cat = df.groupby(col)['is_churned'].agg(['mean', 'count']).sort_values('mean', ascending=False)
    churn_by_cat.columns = ['Churn Rate', 'Count']
    print(f'\n{h_id}: {title}')
    print(f'  Chi-square: {chi2:.2f}, dof: {dof}, p-value: {pval:.2e}')
    print(f'  Result: {"SIGNIFICANT" if sig else "NOT SIGNIFICANT"}')
    print(f'  Churn rate by {col}:')
    for idx, row in churn_by_cat.iterrows():
        print(f'    {str(idx):25s} {row["Churn Rate"]:.1%}  (n={int(row["Count"]):,})')
    hypothesis_results.append({'ID': h_id, 'Hypothesis': title, 'Test': 'Chi-Square',
                               'p-value': f'{pval:.2e}', 'Significant': 'Yes' if sig else 'No',
                               'Verdict': 'SUPPORTED' if sig else 'NOT SUPPORTED'})
    return sig

## 2.1 Load data

In [ ]:
retention = load_dataframe(os.path.join(PROCESSED_DATA_DIR, 'retention_cleaned.csv'))
bob = load_dataframe(os.path.join(PROCESSED_DATA_DIR, 'bob_cleaned.csv'))

# merge minimal BoB data for H10
bob_agg = bob.groupby('account_number').agg(
    bob_total_revenue=('total_bob', 'sum'),
    bob_num_products=('product_name', 'nunique'),
).reset_index()
df = retention.merge(bob_agg, left_on='Customer Account Number', right_on='account_number', how='left')

print(f'\nWorking dataset: {df.shape}')
print(f'Churn rate: {df["is_churned"].mean():.1%}')

## 2.2 Target distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = df['is_churned'].value_counts()
labels = ['Customer Saved', 'Customer Lost']
colors = ['#2ecc71', '#e74c3c']

axes[0].bar(labels, counts.values, color=colors, edgecolor='black')
axes[0].set_title('Target Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90, explode=[0, 0.05])
axes[1].set_title('Churn Rate')
plt.tight_layout()
plt.show()

## 2.3 H1: Higher VAN customers are less likely to churn

In [ ]:
test_numeric(df, 'VAN', 'H1', 'Higher VAN customers are less likely to churn')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, color, name in [(0, '#2ecc71', 'Saved'), (1, '#e74c3c', 'Lost')]:
    data = df[df['is_churned'] == label]['VAN'].dropna()
    axes[0].hist(data, bins=50, alpha=0.6, color=color, label=name, edgecolor='black')
axes[0].set_title('VAN Distribution by Churn Status')
axes[0].legend()
df.boxplot(column='VAN', by='is_churned', ax=axes[1])
axes[1].set_title('VAN by Churn Status')
plt.tight_layout()
plt.show()

## 2.4 H2: More machines/contracts → lower churn

In [ ]:
test_numeric(df, 'Machines', 'H2a', 'More machines → lower churn')
test_numeric(df, 'Number of Contracts', 'H2b', 'More contracts → lower churn')

## 2.5 H3: Certain case titles predict churn

In [ ]:
test_categorical(df, 'churn_reason_group', 'H3', 'Certain case titles predict churn more strongly')

fig, ax = plt.subplots(figsize=(10, 5))
churn_rate = df.groupby('churn_reason_group')['is_churned'].mean().sort_values(ascending=False) * 100
churn_rate.plot(kind='bar', ax=ax, color='#e74c3c', edgecolor='black', alpha=0.8)
ax.set_title('Churn Rate by Churn Reason Group')
ax.set_ylabel('Churn Rate (%)')
ax.axhline(y=df['is_churned'].mean() * 100, color='black', linestyle='--', label=f'Overall: {df["is_churned"].mean()*100:.1f}%')
ax.legend()
plt.tight_layout()
plt.show()

## 2.6 H4: Risk type is a strong predictor of churn

In [ ]:
test_categorical(df, 'Risk', 'H4', 'Risk type is a strong predictor of churn')

## 2.7 H5: Larger companies have lower churn rates

In [ ]:
test_categorical(df, 'CompanySize', 'H5', 'Larger companies have lower churn rates')

## 2.8 H6: Higher-tier customers churn less

In [ ]:
test_categorical(df, 'Customer Tier', 'H6', 'Higher-tier customers churn less')

## 2.9 H7: Proactive case origins lead to lower churn

In [ ]:
test_categorical(df, 'Case Origin', 'H7', 'Proactive case origins lead to lower churn')

## 2.10 H8: Overdue services/repair cases → higher churn

In [ ]:
test_numeric(df, 'Number of OverdueServices', 'H8a', 'Overdue services → higher churn')
test_numeric(df, 'Number Of Repair Cases', 'H8b', 'More repair cases → higher churn')

## 2.11 H9: Full pulls are more associated with churn

In [ ]:
test_categorical(df, 'Pull Type', 'H9', 'Full pulls are more associated with churn')

## 2.12 H10: Higher BoB revenue customers churn less

In [ ]:
# only test on rows that have BoB data
df_bob = df[df['bob_total_revenue'].notna()].copy()
test_numeric(df_bob, 'bob_total_revenue', 'H10', 'Higher BoB revenue customers churn less')

## 2.13 Correlation matrix

In [ ]:
# correlation heatmap for numeric features vs churn
num_cols = [c for c in NUMERIC_FEATURES if c in df.columns] + ['is_churned']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, ax=ax)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 2.14 Hypothesis results summary

In [ ]:
results_df = pd.DataFrame(hypothesis_results)
display(results_df)